# Vue2 + Vue3 框架 · 第 06 天

## Vue Router 路由进阶与面经基础版

> 本篇依据第 06 天 MD 笔记、PPT 和随堂代码整理。重点是声明式导航、路由传参、编程式导航、嵌套路由、路由缓存和工程规范。

## 一、知识地图

| 内容 | 关键 API | 解决的问题 |
| --- | --- | --- |
| 声明式导航 | `router-link`、高亮类名 | 导航跳转和当前项高亮 |
| 路由传参 | `query`、动态路由 `params` | 列表页把条件或 id 带到下一页 |
| 基础配置 | 重定向、404、`mode` | 默认入口、异常路径和 URL 形式 |
| 编程式导航 | `this.$router.push/replace/go` | 点击按钮或请求完成后跳转 |
| 嵌套路由 | `children`、嵌套 `router-view` | 只更新页面局部内容 |
| 页面缓存 | `<keep-alive>`、`activated` | 保留列表滚动位置和筛选状态 |
| 工程规范 | Vue CLI、ESLint | 创建项目和统一代码质量 |

牢记：`$router` 是路由器对象，负责跳转；`$route` 是当前路由信息，负责读取路径、参数和元信息。

## 二、声明式导航与高亮

`router-link` 是 Vue Router 提供的导航组件，使用 `to` 指定路径，不需要写 `#`：

```vue
<router-link to="/find">发现音乐</router-link>
<router-link to="/my">我的音乐</router-link>
<router-view />
```

它会自动生成链接并根据当前路径添加高亮类名：

- `router-link-active`：模糊匹配。`to="/my"` 可以匹配 `/my`、`/my/detail` 等。
- `router-link-exact-active`：精确匹配，只匹配 `/my`。

```css
.router-link-active { color: #f00; }
.router-link-exact-active { font-weight: bold; }
```

如果需要更短的类名，可以在路由对象中配置：

```js
const router = new VueRouter({
  linkActiveClass: 'active',
  linkExactActiveClass: 'exact-active',
  routes: []
})
```

## 三、声明式导航传参

### 3.1 查询参数

适合同时传递多个可选条件：

```vue
<router-link to="/search?words=Vue&page=1">搜索 Vue</router-link>
```

目标页面通过 `$route.query` 获取：

```js
created () {
  console.log(this.$route.query.words)
  console.log(this.$route.query.page)
}
```

### 3.2 动态路由参数

适合路径中有一个明确资源 id：

```js
// router/index.js
{ path: '/detail/:id', component: ArticleDetail }
```

```vue
<router-link :to="`/detail/${item.id}`">查看详情</router-link>
```

```js
created () {
  const id = this.$route.params.id
}
```

如果参数可以省略，使用可选符：`path: '/search/:words?'`。查询参数一般更适合多个筛选条件，动态路由更简洁、语义更清楚。

## 四、重定向、404 与路由模式

### 4.1 默认入口重定向

```js
const routes = [
  { path: '/', redirect: '/home' },
  { path: '/home', component: Home }
]
```

### 4.2 404 路由

通配路由放在其他规则最后：

```js
{ path: '*', component: NotFound }
```

### 4.3 hash 与 history

```js
const router = new VueRouter({
  mode: 'history',
  routes
})
```

- `hash` 默认使用 `/#/home`，刷新时通常不需要服务器额外配置。
- `history` 使用 `/home`，上线时服务器必须把未知路径回退到 `index.html`，否则刷新会 404。

## 五、编程式导航

点击按钮、请求成功或权限判断后，需要在 JavaScript 中跳转：

```js
// 路径跳转
this.$router.push('/home')
this.$router.push({ path: '/home' })

// 命名路由（路径较长时更清晰）
this.$router.push({ name: 'article-detail' })

// 返回和替换
this.$router.go(-1)
this.$router.replace('/login')
```

路由规则配置名称：

```js
{ name: 'article-detail', path: '/detail/:id', component: ArticleDetail }
```

编程式导航传参：

```js
// query
this.$router.push({
  path: '/search',
  query: { words: this.words, page: 1 }
})

// 动态路由
this.$router.push({ path: `/detail/${id}` })

// 命名路由 + params
this.$router.push({ name: 'article-detail', params: { id } })
```

注意：`path` 不能和 `params` 同时使用；需要 `params` 时使用命名路由。

## 六、嵌套路由与面经基础版

当页面有固定外壳，只有中间内容切换时，使用嵌套路由：

```js
const routes = [
  {
    path: '/',
    component: Layout,
    redirect: '/article',
    children: [
      { path: 'article', component: Article },
      { path: 'collect', component: Collect },
      { path: 'like', component: Like },
      { path: 'user', component: User }
    ]
  },
  { path: '/detail/:id', component: ArticleDetail }
]
```

父级 `Layout.vue` 必须提供二级出口：

```vue
<template>
  <div class="layout">
    <main><router-view /></main>
    <nav>
      <router-link to="/article">面经</router-link>
      <router-link to="/collect">收藏</router-link>
      <router-link to="/like">喜欢</router-link>
      <router-link to="/user">我的</router-link>
    </nav>
  </div>
</template>
```

二级路由的 `path` 可以写成相对路径；关键是它们要渲染到哪个父组件的 `<router-view>` 中。

## 七、面经列表与详情

列表页通常在 `created` 中请求数据，保存到响应式数组，再用 `v-for` 渲染：

```js
import axios from 'axios'
export default {
  data () { return { articleList: [] } },
  async created () {
    const { data: { result: { rows } } } = await axios.get(
      'https://mock.boxuegu.com/mock/3083/articles'
    )
    this.articleList = rows
  }
}
```

点击列表项，把 id 传给详情页：

```js
this.$router.push({ path: '/detail', query: { id: item.id } })
```

详情页读取参数后请求详情接口：

```js
async created () {
  const id = this.$route.query.id
  const { data } = await axios.get(`/articles/${id}`)
  this.detail = data
}
```

如果使用动态路由，则读取 `this.$route.params.id`，并在路由规则中写 `/detail/:id`。

## 八、`keep-alive` 缓存组件

列表页切到详情页再返回时，组件默认会销毁并重新创建，可能丢失滚动位置和筛选条件。使用 `keep-alive` 缓存：

```vue
<keep-alive include="Article">
  <router-view />
</keep-alive>
```

被缓存组件会多出两个钩子：

- `activated`：组件从缓存中重新激活。
- `deactivated`：组件被缓存、暂时离开页面。

```js
export default {
  activated () { console.log('回到列表，可恢复滚动位置') },
  deactivated () { console.log('离开列表，暂停轮询') }
}
```

`include` 匹配组件的 `name`，因此页面组件要设置 `name`。不要缓存所有页面，表单和大列表可以按需缓存。

## 九、Vue CLI 自定义创建与 ESLint

```bash
vue create vue-router-demo
```

创建时可选择 Babel、Router、CSS 预处理器和 ESLint。ESLint 的作用是统一代码风格并尽早发现错误。

```bash
npm run lint
```

常见修复点：

- 变量和组件命名遵循项目约定。
- 删除未使用的变量和导入。
- 语句末尾、缩进和引号按项目配置执行。
- 不要为了通过检查关闭规则；先修复真实问题。

编辑器安装 ESLint 插件后可以实时提示，提交前再执行脚本确认。

## 十、当天速记与练习

- 声明式导航用 `router-link`，显示出口用 `router-view`。
- 读取路由信息用 `$route`，跳转用 `$router`。
- 查询参数从 `$route.query` 读取，动态参数从 `$route.params` 读取。
- `push` 会保留历史记录，`replace` 会替换当前记录，`go(-1)` 返回上一页。
- 嵌套路由需要 `children` 和父级 `<router-view>`。
- 列表返回详情时可用 `keep-alive` 保留页面状态。

练习：完成面经首页、详情页、底部二级导航、404 页面，并分别用 query 和动态路由实现一次详情跳转。